In [1]:
import os
import pandas as pd
import ast
from Bio import SeqIO
from tqdm import tqdm
import rpy2.robjects as robjects
import warnings

warnings.filterwarnings('ignore')

def max_dict(dic):
    max_num = None
    for key in dic:
        try:
            int(max_num)
        except:
            max_num = dic[key]
        if dic[key] >= max_num:
            max_key = key
            max_num = dic[key]
    return max_key, max_num

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
def accession_data(acc_n, chr_data, pla_data, file_folder, label):
    acc_size = sum((chr_data | pla_data).values())
    bitscore_csv = pd.read_csv(f'{file_folder}/{acc_n}/sum_replicon_bitscore.csv', index_col = 0)
    bitscore_csv['normalized_by_size'] = bitscore_csv[f'{acc_n}-{label}']/acc_size
    bitscore_pd = bitscore_csv[f'{acc_n}-{label}'].to_frame().set_axis([f'{acc_n}'], axis=1).T
    normalized_pd = bitscore_csv['normalized_by_size'].to_frame().set_axis([f'{acc_n}'], axis=1).T
    return bitscore_pd, normalized_pd

def MS_replicon_data(acc_n, chr_data, pla_data, file_folder, label, MS_replicon):
    max_key, max_num = max_dict(chr_data | pla_data)
    bitscore_csv = pd.read_csv(f'{file_folder}/{acc_n}/replicon_bitscore.csv', index_col = 0)
    bitscore_csv['normalized_by_size'] = bitscore_csv[f'{max_key}-{label}']/max_num
    bitscore_csv = bitscore_csv[bitscore_csv.index.isin(MS_replicon['accession'])]
    bitscore_pd = bitscore_csv[f'{max_key}-{label}'].to_frame().set_axis([f'{acc_n}-{max_key}'], axis=1).T
    normalized_pd = bitscore_csv['normalized_by_size'].to_frame().set_axis([f'{acc_n}-{max_key}'], axis=1).T
    return bitscore_pd, normalized_pd

def typical_chrom_data(acc_n, chr_data, pla_data, file_folder, label, typical_chr):
    merged_data = chr_data | pla_data
    acc_list = typical_chr['accession'].to_list()
    sum_size = 0
    merged_replicons = []
    for replicon in merged_data:
        if f'{acc_n}-{replicon}' in acc_list:
            sum_size += merged_data[replicon]
            merged_replicons.append(replicon)
    bitscore_csv = pd.read_csv(f'{file_folder}/{acc_n}/replicon_bitscore.csv', index_col = 0)
    bitscore_csv['prefix'] = bitscore_csv.index.str.split('-').str[0]
    bitscore_csv = bitscore_csv.groupby('prefix').sum()
    target_cols = [f"{rep}-{label}" for rep in merged_replicons]
    bitscore_csv[f'{acc_n}-{label}'] = bitscore_csv[target_cols].sum(axis=1)
    bitscore_csv['normalized_by_size'] = bitscore_csv[f'{acc_n}-{label}']/sum_size
    bitscore_pd = bitscore_csv[f'{acc_n}-{label}'].to_frame().set_axis([f'{acc_n}'], axis=1).T
    normalized_pd = bitscore_csv['normalized_by_size'].to_frame().set_axis([f'{acc_n}'], axis=1).T
    return bitscore_pd, normalized_pd
    
def draw_tree(folder, label, type_name, method = 'average'):
    robjects.r('library("ape")')
    robjects.r(f'setwd("{folder}")')
    robjects.r(f'da <- read.csv(file = "{type_name}_bitscore-{label}.csv", row.names = 1, header=T)')
    robjects.r('da <- dist(da)')
    robjects.r(f'hc <- do.call("hclust", list(da, method="{method}"))')
    robjects.r('tr <- as.phylo(hc)')
    robjects.r(f'write.tree(tr, file = "tr_{method}-{type_name}-{label}.txt",)')
    robjects.r(f'da <- read.csv(file = "{type_name}_bitscore-{label}-normalized_by_size.csv", row.names = 1, header=T)')
    robjects.r('da <- dist(da)')
    robjects.r(f'hc <- do.call("hclust", list(da, method="{method}"))')
    robjects.r('tr <- as.phylo(hc)')
    robjects.r(f'write.tree(tr, file = "tr_{method}-{type_name}-{label}-normalized_by_size.txt",)')

In [3]:
def prepare_tree_data(genus_name, label, org_data_n, file_folder, folder, type_name, add_pd=None):
    with tqdm(total = len(org_data_n), desc=f'{type_name}-{genus_name}-{label}', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        temp_table = []
        temp_nomalized = []
        for i in org_data_n.index:
            acc_n = org_data_n['accession'][i]
            chr_data = ast.literal_eval(org_data_n['chromosome contigs'][i])
            pla_data = ast.literal_eval(org_data_n['plasmid contigs'][i])
            if type_name == 'accession':
                bitscore_pd, normalized_pd = accession_data(acc_n, chr_data, pla_data, file_folder, label)
            elif type_name == 'MS_replicon':
                bitscore_pd, normalized_pd = MS_replicon_data(acc_n, chr_data, pla_data, file_folder, label, add_pd)
            elif type_name == 'typical_chr_by_acc':
                bitscore_pd, normalized_pd = typical_chrom_data(acc_n, chr_data, pla_data, file_folder, label, add_pd)
            temp_table.append(bitscore_pd)
            temp_nomalized.append(normalized_pd)
            pbar.update(1)
        score_table = pd.concat(temp_table).fillna(0)
        score_nomalized = pd.concat(temp_nomalized).fillna(0)
    os.chdir(folder)
    score_table.to_csv(f'{type_name}_bitscore-{label}.csv')
    score_nomalized.to_csv(f'{type_name}_bitscore-{label}-normalized_by_size.csv')

    %time draw_tree(folder, label, type_name, method = 'average')

In [4]:
labels = ['original', 'pident_90', 'pident_95']
label = labels[1]

for genus_name in keep_genus:
    org_data_n = all_data[all_data['genus_clean'].str.contains(genus_name, na=False)].reset_index(drop=True)
    file_folder = f'/active-data/analysis_results/chr_pla/genus/cor-pla_fraction_records/{genus_name}'
    folder = f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}/tree_data'
    if not os.path.exists(folder):
        os.makedirs(folder)
    os.chdir(folder)
    replicon_info = pd.read_csv(f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
    MS_replicon, NMS_replicon = replicon_info[replicon_info['ms-label'] == 'MS_replicon'], replicon_info[replicon_info['ms-label'] == 'NMS_replicon']
    typical_chr = replicon_info[replicon_info[f'category-{label}'] == 'typical chromosome']
    
    prepare_tree_data(genus_name, label, org_data_n, file_folder, folder, type_name='accession')
    prepare_tree_data(genus_name, label, org_data_n, file_folder, folder, type_name='MS_replicon', add_pd=MS_replicon)
    prepare_tree_data(genus_name, label, org_data_n, file_folder, folder, type_name='typical_chr_by_acc', add_pd=typical_chr)

accession-Escherichia-pident_90: 100%|███████████████████████████| 4.20k/4.20k [00:39<00:00, 106B/s]


CPU times: user 17min 28s, sys: 2.68 s, total: 17min 31s
Wall time: 17min 32s


MS_replicon-Escherichia-pident_90: 100%|████████████████████████| 4.20k/4.20k [02:23<00:00, 29.4B/s]


CPU times: user 21min 56s, sys: 2.52 s, total: 21min 58s
Wall time: 22min


typical_chr_by_acc-Escherichia-pident_90: 100%|█████████████████| 4.20k/4.20k [03:35<00:00, 19.5B/s]


...

CPU times: user 662 ms, sys: 7.96 ms, total: 670 ms
Wall time: 690 ms


MS_replicon-Helicobacter-pident_90: 100%|████████████████████████████| 416/416 [00:01<00:00, 361B/s]


CPU times: user 857 ms, sys: 6.96 ms, total: 863 ms
Wall time: 879 ms


typical_chr_by_acc-Helicobacter-pident_90: 100%|█████████████████████| 416/416 [00:01<00:00, 230B/s]


CPU times: user 651 ms, sys: 9.98 ms, total: 661 ms
Wall time: 676 ms
